# Logistic Regression with scikit-learn

**Notebook 4 of 6.** Notebook 03 took the model apart on one species and one measurement.
It ended on a limit: a body-mass boundary that suits Adelie penguins is 1,340 g off for
Gentoos. This notebook lifts the limit in two ways.

- **More features.** All four measurements at once, then the species too, for all 333
  penguins.
- **More classes.** Predicting the species itself, a target with three classes.

The mechanics stay as in 03: create, fit, predict. What is new is reading a model with
several coefficients, and why they first have to be put on a common scale.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

sns.set_style("whitegrid")

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")
labeled = df_penguins.dropna(subset=["sex"])

measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

## 1. The workflow, with four features

The same split as in notebook 02, so the scores can be compared with the baselines there:
**50 percent** for the majority guess and **71 percent** for the best body-mass
threshold.

In [ ]:
X = labeled[measurements]
y = labeled["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

print(f"train accuracy: {accuracy_score(y_train, model.predict(X_train)):.3f}")
print(f"test accuracy:  {accuracy_score(y_test, model.predict(X_test)):.3f}")

**88 percent on penguins the model has never seen,** from 71 with one measurement. The
target is text this time, `FEMALE` and `MALE`, and scikit-learn handles that itself:
`classes_` shows the order, and the second class is the one the probabilities and
coefficients refer to.

In [ ]:
model.classes_

Four features give four coefficients, one per column, in the order of the columns:

In [ ]:
pd.Series(model.coef_[0], index=measurements).round(3)

**These numbers cannot be compared with each other.** Each is "change in `t` per unit",
and the units differ: bill depth is in millimeters and varies by a few, body mass is in
grams and varies by thousands. The body-mass coefficient looks smallest only because a
single gram is a tiny step.

## 2. A common scale

`StandardScaler` rescales every column to mean 0 and standard deviation 1. A value of 1
then means "one standard deviation above average", whatever the original unit, and the
coefficients become comparable: each is the effect of one standard deviation of its
feature.

**The scaler is fitted on the training set only**, and then applied to both sets. It
learns a mean and a standard deviation from the data it is fitted on, and those must not
come from the test penguins. Notebook 05 shows what goes wrong otherwise.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean and std, then apply
X_test_scaled = scaler.transform(X_test)         # apply only, learn nothing

model_scaled = LogisticRegression()
model_scaled.fit(X_train_scaled, y_train)

print(f"test accuracy: {accuracy_score(y_test, model_scaled.predict(X_test_scaled)):.3f}")
pd.Series(model_scaled.coef_[0], index=measurements).round(2)

The accuracy barely moves, since scaling changes the units, not the information. But now
the coefficients can be read:

- **Bill depth and body mass carry most of the weight**, about equally. Both are positive:
  deeper bills and heavier bodies point to male.
- **Bill length adds a little**, and **flipper length is close to zero**. Once the other
  three are known, flipper length tells the model almost nothing new about sex.

> **Exercise:**
>
> Fit the scaled model on **only bill depth and body mass**, the two features with the
> largest coefficients. Remember to fit a new scaler on the new training columns. How much
> test accuracy is lost compared with all four?

In [ ]:
# your code: the scaled model on two features

<details><summary>
Click here to compare your answer...
</summary>

The two features alone reach about the same test accuracy as all four. That is what the
coefficients said: the other two add little. A large coefficient on scaled features is a
reasonable first guide to what matters, though not a proof; notebook 02 showed that on 84
test penguins a couple of points is within the noise.
</details>

## 3. Adding the species

Notebook 03 showed that the same mass means different things for different species. The
species is a text column, and a model only takes numbers. The standard fix is **one-hot
encoding**: one new column per species, holding 1 for penguins of that species and 0 for
the others.

In [ ]:
X_species = pd.get_dummies(labeled[measurements + ["species"]], columns=["species"], dtype=int)
X_species.head()

> **Exercise:**
>
> Split `X_species` with the same settings as before, scale it, fit a logistic regression,
> and compare its test accuracy with the model on four measurements. Which of the three
> species columns gets the largest coefficient, and in which direction?

In [ ]:
# your code: measurements plus species

<details><summary>
Click here to compare your answer...
</summary>

Test accuracy rises to about **93 percent**. The largest species coefficient belongs to
`species_Adelie`, and it is **positive**: at a given mass and bill size, an Adelie is more
likely to be male, because Adelie penguins are the smallest and a large Adelie stands out.
The model now does what the single boundary in 03 could not: it moves its judgment
depending on the species.
</details>

## 4. Three classes: predicting the species

The second question from notebook 01: **which species is a penguin?** The code does not
change at all. scikit-learn sees three distinct values in the target and fits a model for
three classes.

Two measurements to begin with, bill length and bill depth, so that the result can be
drawn.

In [ ]:
bill = ["bill_length_mm", "bill_depth_mm"]
X = labeled[bill]
y = labeled["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("classes:", model.classes_)
print(f"baseline, test: {(y_test == y_train.mode()[0]).mean():.3f}")
print(f"test accuracy:  {accuracy_score(y_test, model.predict(X_test_scaled)):.3f}")

From a baseline of 44 percent, always guessing Adelie, to about 95. `predict_proba` now
returns **three columns**, one per class in the order of `classes_`, and each row sums to
1:

In [ ]:
proba = pd.DataFrame(model.predict_proba(X_test_scaled), columns=model.classes_)
proba["sum"] = proba.sum(axis=1)
proba.head().round(3)

The model gives each class its own line `t`, with its own coefficients, which is why
`coef_` now has one row per class. Each penguin goes to the class with the highest
probability, and the result is a map of which bill shape means which species. The code
below colors every point of a fine grid by the class the model predicts there.

In [ ]:
# a fine grid of bill shapes, in the original units
lengths, depths = np.meshgrid(np.linspace(30, 62, 300), np.linspace(12, 23, 300))
grid = pd.DataFrame({"bill_length_mm": lengths.ravel(), "bill_depth_mm": depths.ravel()})

# the predicted class at every grid point, as a number for coloring
predicted = model.predict(scaler.transform(grid))
codes = pd.Categorical(predicted, categories=model.classes_).codes.reshape(lengths.shape)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.contourf(lengths, depths, codes, levels=[-0.5, 0.5, 1.5, 2.5],
            colors=["#fbd3a6", "#c8b6e2", "#a8dcd1"], alpha=0.6)
sns.scatterplot(data=labeled.loc[X_train.index], x="bill_length_mm", y="bill_depth_mm",
                hue="species", hue_order=model.classes_,
                palette=["darkorange", "purple", "teal"], ax=ax)
ax.set_title("Where the model predicts each species, with the training penguins");

The borders between the regions are straight lines: logistic regression draws straight
boundaries, always. Here that is enough, because the three species form three clouds with
little overlap.

> **Exercise:**
>
> Find the test penguin the model is **least sure about**: the one whose highest
> probability is lowest. Print its measurements, its true species and its three
> probabilities. Where does it sit on the map above?

<details><summary>
Click here for a hint...
</summary>

`proba[model.classes_].max(axis=1)` gives each penguin's highest probability, and
`.idxmin()` the position of the lowest one. `X_test.iloc[position]` and
`y_test.iloc[position]` give that penguin's measurements and species.
</details>

In [ ]:
# your code: the test penguin the model is least sure about

<details><summary>
Click here to compare your answer...
</summary>

It is a **Chinstrap** with a bill 43.2 mm long and 16.6 mm deep, and the model spreads
its belief almost evenly: about 0.29 Adelie, 0.32 Chinstrap, 0.39 Gentoo. It sits close to
the point where the three regions meet, where the model is least sure by construction.
Here the narrow favorite is wrong, so this penguin is misclassified as a Gentoo. A low top
probability is a warning worth reading: the model is telling you it is guessing.
</details>

> **Exercise:**
>
> Fit the species model on **all four measurements** instead of two, scaling as before.
> What test accuracy does it reach?

In [ ]:
# your code: species from all four measurements

<details><summary>
Click here to compare your answer...
</summary>

**Every test penguin classified correctly**, and 99 percent on the training set. Flipper
length and body mass separate the Gentoos, which are much larger, and bill length and
depth separate Adelies from Chinstraps. Two measurements drew the map; four leave almost
no penguin in doubt.

`model.score(X, y)` in this solution is a shortcut: for a classifier it is the accuracy
of `model.predict(X)` against `y`.
</details>

## Summary

| Model | Test accuracy |
| :-- | --: |
| Sex, majority baseline | 50% |
| Sex, body-mass threshold (notebook 02) | 71% |
| Sex, four measurements | 88% |
| Sex, four measurements and species | 93% |
| Species, majority baseline | 44% |
| Species, bill length and depth | 95% |
| Species, four measurements | 100% |

- The workflow is always the same: **split, create, fit, predict, score** on the test
  set.
- **Coefficients are only comparable on a common scale.** `StandardScaler` provides it,
  and is **fitted on the training set only**, then applied to both.
- A text feature enters the model through **one-hot encoding**, `pd.get_dummies`.
- With three classes, `predict_proba` has three columns that sum to 1, `coef_` has one row
  per class, and the boundaries between classes are straight lines.

Next, in `05_leakage.ipynb`: two ways a model can look better than it is, because some
information about the answer slipped in where it should not have.